# 00 · Start here — look at the data

Run the cells top to bottom (Shift+Enter). Works the same on your laptop, on Kaggle and on Colab.

* **Dev subset** (2% of Source-1 entities + all their matches + 2% of distractors) is used for exploring — fast, fits in RAM.
* The full data is only loaded by the pipeline, via DuckDB/Parquet.
* Scores are always computed with `erharness` so everyone's numbers are comparable.

In [1]:
# --- setup: make `amlc` and `erharness` importable from anywhere in the repo ---
import sys, pathlib
REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "amlc").exists())
for p in (REPO / "src", REPO):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import pandas as pd
pd.set_option("display.max_colwidth", 80, "display.width", 200)
from amlc import paths, data
print("raw data :", paths.raw_dir())
print("work dir :", paths.work_dir())

raw data : C:\Users\Harsh Vardhan\Downloads\6ab10eb3b23ba_student_resource\student_resource\dataset
work dir : C:\Users\Harsh Vardhan\projects\amlc-er\data


## 1 · Convert to Parquet + build the dev subset (first run only, ~5–15 min on a laptop)

In [2]:
import os
if not os.path.exists(data.pq("test_source3")):
    data.convert()
if not os.path.exists(data.pq("train_source1", dev=True)):
    data.make_dev()
print(open(paths.work_dir() / "profile.md").read() if (paths.work_dir() / "profile.md").exists() else data.profile())

# Data profile

| file | rows | countries | empty name | empty address | Devanagari name | median name len | median addr len |
|---|---|---|---|---|---|---|---|
| train/train_source1 | 2,206,821 | US:1323633 India:883188 | 0.000% | 0.000% | 0.00% | 24 | 41 |
| train/train_source2 | 5,034,616 | US:3016817 India:2017799 | 0.000% | 3.356% | 5.35% | 25 | 37 |
| train/train_source3 | 5,285,603 | US:3170056 India:2115547 | 0.000% | 3.328% | 2.99% | 25 | 42 |
| test/test_source1 | 1,732,544 | India:809986 US:663106 France:259452 | 0.000% | 0.000% | 0.00% | 24 | 50 |
| test/test_source2 | 4,887,273 | India:2312565 US:1871330 France:703378 | 0.000% | 2.648% | 6.32% | 25 | 43 |
| test/test_source3 | 5,082,316 | India:2405000 US:1945701 France:731615 | 0.000% | 2.678% | 3.56% | 25 | 43 |

## Matches per Source-1 entity (train)

| n matches | entities | share |
|---|---|---|
| 0 | 123,247 | 5.585% |
| 1 | 119,157 | 5.399% |
| 2 | 375,212 | 17.002% |
| 3 | 530,841 | 24.055% |
| 4 | 484,115 | 21.937

## 2 · Load the dev subset

In [3]:
gt = data.load("train_ground_truth", dev=True)
s1 = data.load("train_source1", dev=True)
s2 = data.load("train_source2", dev=True)
s3 = data.load("train_source3", dev=True)
recs = pd.concat([s1, s2, s3]).set_index("entity_id")
gt["matches"] = gt.matched_entity_ids.map(lambda x: [t for t in x.split(",") if t])
gt["n"] = gt.matches.map(len)
print({k: len(v) for k, v in dict(gt=gt, s1=s1, s2=s2, s3=s3).items()})
gt.n.value_counts().sort_index()

{'gt': 44215, 's1': 44215, 's2': 101198, 's3': 105813}


n
0      2485
1      2332
2      7449
3     10749
4      9612
5      6432
6      3383
7      1274
8       410
9        74
10       15
Name: count, dtype: int64

## 3 · Read real match groups — this is where the normalisation rules come from

In [4]:
def show(s1_id):
    row = gt.loc[gt.source1_entity_id == s1_id].iloc[0]
    ids = [s1_id] + row.matches
    return recs.loc[[i for i in ids if i in recs.index], ["business_name", "business_address", "country"]]

country = s1.set_index("entity_id").country
for c in ["US", "India"]:
    pool = gt[(gt.source1_entity_id.map(country) == c) & (gt.n > 0)]
    for sid in pool.sample(min(3, len(pool)), random_state=1).source1_entity_id:
        display(show(sid))

,business_name,business_address,country
entity_id,,,
S1-964786053,Fairfax County Heart Innovative Institute Inc,"6743 Rock Brook Drive, Fairfax County, VA",US
S2-259516490,fairfaxcountyheart.com #72386,"6743-C ROCK BROOK DR, CLIFTON, VA",US
S2-987270875,Fairfax County Heart Innovative Institute Incorporated,"6743-C ROCK BROOK DRIVE, FAIRFAX COUNTY, VA",US
S3-664461540,Fairfax County Heart Innovative Institute INC,"6743 Rock Brook Dr, Fairfax County, Virginia",US
S3-305203352,Veosol D.B.A. Fairfax County Heart Innovative Institute Inc,"6743 Rock Brook Drive, Fairfax County, VA",US


,business_name,business_address,country
entity_id,,,
S1-430016384,Joh Frontier Liquor,"1235 Berryhill Lane, Rock Hill, SC",US
S2-936422797,JOH FRONTIER (LIQUOR),"1235 BERRYHILL LN, ROCK HILL, SC",US
S3-961745619,Joh-Frontier Liquor,,US
S3-469648002,Joh Frontier Liquor Corp,"#1235 Berryhill Ln, ROCK HILL, South Carolina",US


,business_name,business_address,country
entity_id,,,
S1-256102717,Womens Health Diamond Care,"239 King Street, Oxford, ME",US
S3-798539459,womenshealthdiamond.com,"Oxford, Maine, 239 King St",US
S3-663305859,Womens Health Diamond Care LLC,"239 King St, Oxford, Maine",US


,business_name,business_address,country
entity_id,,,
S1-298181543,Indo Construction Private Limited,"S/F-3, Prop 1736, Sher Singh Bazar Kotla Mubarakpur, New Delhi, Delhi",India
S2-400393739,INDO CONSTRUCTION,"#5-17 S/F-3, PROP 1736, SHER SINGH BAZAR KOTLA MUBARAKPUR, NEW DELHI, Delhi",India
S2-353219614,INDO CONSTRUCTION PRIVATE LÍMITED,"5-17 S/F-3, PROP 1736, SHER SINGH BAZAR KOTLA MUBARAKPUR, NEW DELHI, दिल्ली",India
S2-234389652,INDO CONSTRUCTION PRIVATE,"H.NO 5-17 S/F-3, PROP 1736, SHER SINGH BAZAR KOTLA MUBARAKPUR, NEW DELHI, Delhi",India
S2-303420396,Indo Private Construction Limited,"DOOR NO 5-17 S/F-3, PROP 1736, SHER SINGH BAZAR KOTLA MUBARAKPUR, NEW DELHI,...",India
S3-577613932,Xylovio,"S/f-3, New Delhi, दिल्ली",India
S3-772160233,Dr Indo Construction Private [Limited],"New Delhi, S/f-3, DL",India
S3-117372927,Indo Construction Private,"New Delhi, S/f-3, New Delhi, DL",India


,business_name,business_address,country
entity_id,,,
S1-262639937,Big Producer Private Limited,"Radha Kunjo, 455 P. K. Guha Road Near Airport Gate No 1, Kolkata, Howrah, We...",India
S2-271937303,বিগ প্রডিউসার প্রাইভেট লিমিটেড,"CALCUTTA, HOWRAH, RADHA KUNJO, 455 P. K. GUHA ROAD NEAR AIRPORT GATE NO 1, প...",India
S3-338766246,Big Producer প্রাইভেট লিমিটেড,"Hn 776 Radha Kunjo, Kolkata, Howrah, WB",India


,business_name,business_address,country
entity_id,,,
S1-854230000,PD Enterprises Pvt. Ltd.,"H No.: 23-22-121, Sivalayam Street, Vijayawada (Urban), Krishna, Andhra Pradesh",India
S2-121763331,PVT. PD ÉNTERPRISES (LTD.),"H NO.: 23-22-121, KRISHNA, VIJAYAWADA (URBAN), Andhra Pradesh",India
S2-175954712,PD ENTERPRISES [PVT.],"Andhra Pradesh, H NO.: 23-22-121, SIVALAYAM STREET, VIJAYAWADA (URBN)",India
S2-227508959,PD Ltd. Pvt. Enterprises,"H NO.: 23-22-121, SIVALAYAM STREET, VIJAYAWADA (URBAN), ఆంధ్రప్రదేశ్",India
S3-448112957,PD Enterprises,"Krishna, AP, Hn 985 H No.: 23-22-121, Vijayawada (Urban)",India
S3-904398801,PD Enterpriscs Pvt. Ltd.,"Hn 985 H No.: 23-22-121, Krishna, Vijayawada (Urban), AP",India


## 4 · Singletons (score 1.0 only if we predict an empty list)

In [5]:
sing = gt[gt.n == 0].source1_entity_id
recs.loc[sing.sample(min(10, len(sing)), random_state=0), ["business_name", "business_address", "country"]]

,business_name,business_address,country
entity_id,,,
S1-715823105,A Cure 7 All,"10414 Peacock Hill Avenue, Unit #52, Gig Harbor, WA",US
S1-996813149,United Gold Hospitality Pvt Ltd,"B4/22, Ground Floor, Dlf Phase 1, Behind Qutub Plaza, Gurgaon, Haryana",India
S1-999366092,Pioneer Hamilton Inc,"1150 Goss Avenue, Village Of Fox Crossing, WI",US
S1-607643897,Great Elm Inc,"51 Highland Street, Unit 1, Peabody, MA",US
S1-207155654,Clear Transit Enterprises LP,"802 Overlake Court, Euless, TX",US
S1-120905380,Primary Care Group,"610 Traci Lane, Swanton, MD",US
S1-858432256,Technologies Lifetime Packaging Private Limited,"Ernakulam, Nagapuzha Kalloorkkad, Ernakulam, Vadakkekunnel, Kerala",India
S1-305240278,American Guild LLC,"773 Lava Pointe Drive, St George City, UT",US
S1-71343846,Logan & Patterson,"18800 Harmony Woods Lane, Germantown, MD",US


## 5 · France — only in the test set, never in train

In [6]:
t1 = pd.read_parquet(data.pq("test_source1"), columns=["business_name", "business_address", "country"])
print(t1.country.value_counts())
fr = t1[t1.country == "France"]
fr.sample(min(15, len(fr)), random_state=0)

country
India     809986
US        663106
France    259452
Name: count, dtype: int64


,business_name,business_address,country
1225253,Shidokan Ecole SARL,"17 Rue de Valmy, Lille, Hauts-de-France",France
1337759,Pique Ecole EURL,"20 BD Jules Verne, Nantes, Pays de la Loire",France
1320154,Chambre Medico SARL,"5 Avenue Gaston Bonnier, Nantes, Pays de la Loire",France
1537396,Supérieure Sante EURL,"9 Rue Rubens, Lille, Hauts-de-France",France
1253696,Vidourle Boulistes Élémentaire SAS,"Pessac, 59 Avenue des Echoppes, Nouvelle-Aquitaine",France
1171124,EHPAD Saint Sophro,"12 RUE de lOrne, 44800, Saint-Herblain, Pays de la Loire",France
1014875,Pipelettes Clinique,"2 Rue Leveux, Calais, Hauts-de-France",France
1403481,Mille Club EURL,"28 Bis Boulevard Joliot Curie, Nantes, Pays de la Loire",France
469017,Pessac Maternelle SARL,"90 Rue Simone de Beauvoir, Pessac, Nouvelle-Aquitaine",France
431706,Communale Club SAS,"Hauts-de-France, 6 BD Papin, Lille",France


## 6 · Scoring sanity check with the harness
All-empty prediction scores exactly the singleton share.

In [7]:
from erharness import metric
gt_sets = {r.source1_entity_id: set(r.matches) for r in gt.itertuples()}
rep = metric.evaluate({k: set() for k in gt_sets}, gt_sets)
print(rep.summary())

macro F0.5        0.05620   (n=44215)
  singleton frac  0.0562
  spec (singletons left empty)     1.0000
  open rate (matched given a set)  0.0000
  F0.5 on opened matched           0.0000
  F0.5 on all matched              0.0000
  pair precision / recall          nan / 0.0000
  exact-set rate on matched        0.0000
  outcomes: gated_shut=41730, singleton_ok=2485
  by n_true: 0:1.0000, 1:0.0000, 2:0.0000, 3+:0.0000
